# LifeJacket Agent 2: Triage & Coordination Agent

This notebook is the **agentic AI / prompt-engineering version** of Agent 2.

Instead of encoding most decision policy as Python `if/else` logic, this notebook builds structured prompt packets for specialist Gemini prompt stages:

- duplicate matching prompt
- severity triage prompt
- recommended action prompt
- dispatch center prompt
- alert card prompt
- guardrail prompt
- audit prompt
- end-of-day documentation

Python is kept intentionally boring: it loads data, formats JSON, displays prompt packets, and shows demo outputs. The reasoning policy lives in prompts.

## 1. GCP JupyterLab Setup

Keep the notebook and its three input/config files in the same JupyterLab folder:

- `agent1_incident_database.csv`
- `agent2_prompt_library.json`
- `agent2_rescue_center_directory.csv`

The setup cell uses the current notebook folder rather than a local computer path. For a different uploaded incident CSV, set the optional `LIFEJACKET_AGENT1_CSV` environment variable before running the notebook.


In [1]:
from pathlib import Path
import json
import os
import pandas as pd

pd.set_option('display.max_columns', 100)
pd.set_option('display.max_colwidth', 180)

PROJECT_DIR = Path.cwd()
PROMPT_LIBRARY_PATH = PROJECT_DIR / 'agent2_prompt_library.json'
RESCUE_CENTER_CSV = PROJECT_DIR / 'agent2_rescue_center_directory.csv'
AGENT1_INPUT_CSV = Path(os.getenv('LIFEJACKET_AGENT1_CSV', str(PROJECT_DIR / 'agent1_incident_database.csv')))

required_project_files = [PROMPT_LIBRARY_PATH, RESCUE_CENTER_CSV]
missing_project_files = [str(path.name) for path in required_project_files if not path.exists()]
assert not missing_project_files, f'Upload required project files into this JupyterLab folder: {missing_project_files}'

prompt_library = json.loads(PROMPT_LIBRARY_PATH.read_text())
prompt_index = {prompt['id']: prompt for prompt in prompt_library['prompts']}
rescue_centers = pd.read_csv(RESCUE_CENTER_CSV).fillna('')

setup_status = pd.DataFrame([
    {'file': PROMPT_LIBRARY_PATH.name, 'available': PROMPT_LIBRARY_PATH.exists(), 'required_for': 'prompt modules'},
    {'file': RESCUE_CENTER_CSV.name, 'available': RESCUE_CENTER_CSV.exists(), 'required_for': 'dispatch selection'},
    {'file': AGENT1_INPUT_CSV.name, 'available': AGENT1_INPUT_CSV.exists(), 'required_for': 'live Agent 1 CSV run'},
])
print('Project directory:', PROJECT_DIR)
setup_status


Project directory: /home/jupyter/Coordination_agent2


,file,available,required_for
0,agent2_prompt_library.json,True,prompt modules
1,agent2_rescue_center_directory.csv,True,dispatch selection
2,agent1_incident_database.csv,True,live Agent 1 CSV run


## 2. Prompt Catalog

These are the prompt stages available to the prompt-first Agent 2 workflow.

## Prompt Library vs. Prompt Execution

Agent 2 contains a library of **28 specialized prompt templates**, but it does not execute every prompt for every incident. The prompt library represents the full set of capabilities available to the Decision Agent, including triage, dispatch, alert generation, safety review, documentation, follow-up, explanation, output repair, and prompt improvement.

For the featured demonstration case, `INC_80ca122f`, Agent 2 executes a focused chain of **10 Gemini prompt calls** (the first two prompts can be combined into Agent 1 as we discussed further):

| Step | Prompt Module | Purpose |
|---:|---|---|
| 1 | `species_group_classifier_few_shot` | Converts Agent 1 species evidence into an operational species group, such as pinniped or cetacean. |
| 2 | `report_validation_prompt` | Checks completeness, missing information, and conflicts in the structured incident report. |
| 3 | `duplicate_matching_prompt` | Compares the current report with earlier incidents using location, time, and species evidence. |
| 4 | `severity_triage_prompt` | Assigns an urgency level and explains the safety indicators that triggered it. |
| 5 | `recommended_action_prompt` | Converts the triage result into an operational response recommendation. |
| 6 | `dispatch_center_prompt` | Selects an appropriate rescue center from the supplied response directory. |
| 7 | `alert_card_generator` | Creates concise coordinator-facing alert information. |
| 8 | `alert_card_guardrail_critic` | Reviews the generated alert for unsafe wording or unsupported claims. |
| 9 | `claim_grounding_auditor` | Verifies that the alert and decisions remain grounded in the incident evidence. |
| 10 | `audit_log_narrator` | Produces an auditable record of the inputs, decisions, triggered rules, and residual risks. |

If end-of-day reporting is generated, Agent 2 executes one additional module:

```text
end_of_day_documentation_prompt

In [2]:
prompt_catalog = pd.DataFrame(prompt_library['prompts'])
prompt_catalog[['id', 'stage', 'model_role', 'purpose', 'allowed_to_change_decisions']]

,id,stage,model_role,purpose,allowed_to_change_decisions
0,species_group_classifier,pre_decision_classification,species group classifier,"Classify the report into the constrained LifeJacket species groups using CV, reporter, and LLM species clues.",False
1,species_group_classifier_few_shot,pre_decision_classification,few-shot species group classifier,Use few-shot examples to classify species group while staying inside the allowed labels.,False
2,prompt_task_router,orchestration,prompt router,Decide which specialist prompt roles should run for a given incident packet.,False
3,json_schema_repair_prompt,output_repair,JSON schema repair prompt,Repair malformed or incomplete model JSON without changing the underlying meaning.,False
4,claim_grounding_auditor,guardrail_validation,claim grounding auditor,Check generated text for unsupported claims against the structured incident packet.,False
5,decision_trace_explainer,audit,decision trace explainer,Create a step-by-step explanation of how the prompt pipeline handled the report.,False
6,coordinator_question_prompt,coordinator_review,coordinator Q&A module,Answer coordinator questions using only the structured incident packet.,False
7,alert_card_tone_adapter,ui_copy_generation,tone adapter,Rewrite approved alert-card copy for a specific audience while preserving facts and decisions.,False
8,incident_validation_reviewer,pre_decision_review,quality-control reviewer,"Review Agent 1 structured output for missing information, contradictions, and follow-up needs before deterministic triage.",False
9,species_consistency_auditor,pre_decision_review,species consistency auditor,Explain whether CV and LLM species outputs appear consistent or require human review.,False


## 3. Demo Incidents

These examples are synthetic and presentation-friendly. They show how different prompt stages would run for different situations.

In [3]:
demo_incidents = pd.DataFrame([
    {
        'incident_id': 'DEMO_001_PINNIPED_WOUND',
        'report_timestamp': '2026-05-20T15:00:00Z',
        'latitude': 34.4208,
        'longitude': -119.6982,
        'place_name': 'Santa Barbara Beach, CA',
        'cv_top1_species': 'California sea lion',
        'cv_top1_confidence': 0.82,
        'cv_top2_species': 'Harbor seal',
        'cv_top2_confidence': 0.18,
        'reported_mobility': 'moving normally',
        'reported_visible_wound': True,
        'reported_bleeding': 'not sure',
        'reported_entanglement': False,
        'reported_near_people': True,
        'reported_near_dogs': False,
        'llm_likely_species': 'California sea lion',
        'llm_species_confidence': 0.95,
        'llm_injury_present': True,
        'llm_wound_present': True,
        'llm_bleeding_present': False,
        'llm_entanglement_present': False,
        'llm_respiratory_distress_present': False,
        'llm_unresponsive_present': False,
        'llm_mobility_concern': 'none',
    },
    {
        'incident_id': 'DEMO_002_DUPLICATE_CRITICAL',
        'report_timestamp': '2026-05-20T15:12:00Z',
        'latitude': 34.4210,
        'longitude': -119.6980,
        'place_name': 'Santa Barbara Beach, CA',
        'cv_top1_species': 'California sea lion',
        'cv_top1_confidence': 0.78,
        'reported_mobility': 'limited',
        'reported_visible_wound': True,
        'reported_bleeding': True,
        'reported_entanglement': False,
        'reported_near_people': True,
        'reported_near_dogs': True,
        'llm_likely_species': 'California sea lion',
        'llm_species_confidence': 0.91,
        'llm_injury_present': True,
        'llm_wound_present': True,
        'llm_bleeding_present': True,
        'llm_entanglement_present': False,
        'llm_respiratory_distress_present': False,
        'llm_unresponsive_present': False,
        'llm_mobility_concern': 'limited',
    },
    {
        'incident_id': 'DEMO_003_CETACEAN_MOBILITY',
        'report_timestamp': '2026-05-20T16:30:00Z',
        'latitude': 34.2746,
        'longitude': -119.2290,
        'place_name': 'Ventura Beach, CA',
        'cv_top1_species': 'dolphin',
        'cv_top1_confidence': 0.74,
        'reported_mobility': 'immobile',
        'reported_breathing_status': 'irregular',
        'reported_visible_wound': False,
        'reported_bleeding': False,
        'reported_entanglement': False,
        'reported_near_people': False,
        'reported_near_dogs': False,
        'llm_likely_species': 'dolphin',
        'llm_species_confidence': 0.86,
        'llm_injury_present': False,
        'llm_wound_present': False,
        'llm_bleeding_present': False,
        'llm_entanglement_present': False,
        'llm_abnormal_posture_present': True,
        'llm_respiratory_distress_present': True,
        'llm_unresponsive_present': False,
        'llm_mobility_concern': 'yes',
    },
    {
        'incident_id': 'DEMO_004_INCOMPLETE_UNKNOWN',
        'report_timestamp': '2026-05-20T17:05:00Z',
        'latitude': None,
        'longitude': None,
        'place_name': 'unknown beach',
        'cv_top1_species': 'unknown',
        'cv_top1_confidence': 0.21,
        'reported_mobility': '',
        'reported_visible_wound': '',
        'reported_bleeding': '',
        'reported_entanglement': '',
        'llm_likely_species': 'unknown',
        'llm_species_confidence': 0.2,
        'llm_needs_followup': True,
        'llm_followup_question': 'Can you share the nearest landmark and whether the animal is moving?',
    },
    {
        'incident_id': 'DEMO_005_TURTLE_ENTANGLEMENT',
        'report_timestamp': '2026-05-20T18:00:00Z',
        'latitude': 33.5427,
        'longitude': -117.7854,
        'place_name': 'Laguna Beach, Orange County, CA',
        'cv_top1_species': 'sea turtle',
        'cv_top1_confidence': 0.88,
        'reported_mobility': 'struggling',
        'reported_visible_wound': False,
        'reported_bleeding': False,
        'reported_entanglement': True,
        'reported_near_people': True,
        'reported_near_dogs': False,
        'llm_likely_species': 'sea turtle',
        'llm_species_confidence': 0.91,
        'llm_injury_present': True,
        'llm_wound_present': False,
        'llm_bleeding_present': False,
        'llm_entanglement_present': True,
        'llm_respiratory_distress_present': False,
        'llm_unresponsive_present': False,
        'llm_mobility_concern': 'yes',
    },
])

demo_incidents

Task was destroyed but it is pending!
task: <Task pending name='Task-86' coro=<_async_in_context.<locals>.run_in_context_pre311() running at /opt/conda/lib/python3.10/site-packages/ipykernel/utils.py:76> cb=[ZMQStream._run_callback.<locals>._log_error() at /opt/conda/lib/python3.10/site-packages/zmq/eventloop/zmqstream.py:563]>


,incident_id,report_timestamp,latitude,longitude,place_name,cv_top1_species,cv_top1_confidence,cv_top2_species,cv_top2_confidence,reported_mobility,reported_visible_wound,reported_bleeding,reported_entanglement,reported_near_people,reported_near_dogs,llm_likely_species,llm_species_confidence,llm_injury_present,llm_wound_present,llm_bleeding_present,llm_entanglement_present,llm_respiratory_distress_present,llm_unresponsive_present,llm_mobility_concern,reported_breathing_status,llm_abnormal_posture_present,llm_needs_followup,llm_followup_question
0,DEMO_001_PINNIPED_WOUND,2026-05-20T15:00:00Z,34.4208,-119.6982,"Santa Barbara Beach, CA",California sea lion,0.82,Harbor seal,0.18,moving normally,True,not sure,False,True,False,California sea lion,0.95,True,True,False,False,False,False,none,NaN,NaN,NaN,NaN
1,DEMO_002_DUPLICATE_CRITICAL,2026-05-20T15:12:00Z,34.4210,-119.6980,"Santa Barbara Beach, CA",California sea lion,0.78,NaN,NaN,limited,True,True,False,True,True,California sea lion,0.91,True,True,True,False,False,False,limited,NaN,NaN,NaN,NaN
2,DEMO_003_CETACEAN_MOBILITY,2026-05-20T16:30:00Z,34.2746,-119.2290,"Ventura Beach, CA",dolphin,0.74,NaN,NaN,immobile,False,False,False,False,False,dolphin,0.86,False,False,False,False,True,False,yes,irregular,True,NaN,NaN
3,DEMO_004_INCOMPLETE_UNKNOWN,2026-05-20T17:05:00Z,NaN,NaN,unknown beach,unknown,0.21,NaN,NaN,,,,,NaN,NaN,unknown,0.20,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,True,Can you share the nearest landmark and whether the animal is moving?
4,DEMO_005_TURTLE_ENTANGLEMENT,2026-05-20T18:00:00Z,33.5427,-117.7854,"Laguna Beach, Orange County, CA",sea turtle,0.88,NaN,NaN,struggling,False,False,True,True,False,sea turtle,0.91,True,False,False,True,False,False,yes,NaN,NaN,NaN,NaN


## 4. Tiny Orchestration Helpers

These helpers do not contain rescue policy. They only prepare JSON context and render prompt templates.

In [4]:
def as_json(value):
    return json.dumps(value, indent=2, ensure_ascii=False, default=str)

def render_prompt(prompt_id, **values):
    return prompt_index[prompt_id]['template'].format(**values)

def row_packet(row):
    return {key: value for key, value in row.to_dict().items() if pd.notna(value)}

def directory_packet(rows=12):
    columns = ['center_name', 'hotline', 'email', 'website', 'response_area', 'response_type']
    return rescue_centers[columns].head(rows).to_dict(orient='records')

def prior_candidate_packet(current_id):
    columns = ['incident_id', 'report_timestamp', 'latitude', 'longitude', 'place_name', 'cv_top1_species', 'llm_likely_species']
    return demo_incidents.loc[demo_incidents['incident_id'] != current_id, columns].head(10).to_dict(orient='records')

/opt/conda/lib/python3.10/asyncio/base_events.py:674: RuntimeWarning: coroutine '_async_in_context.<locals>.run_in_context_pre311' was never awaited
  self._ready.clear()


## 5. Build A Prompt-First Agent 2 Packet

This packet is what the app would send through the Gemini prompt stages.

In [5]:
def build_prompt_first_packet(row):
    incident = row_packet(row)
    species_context = {
        'cv_top1_species': incident.get('cv_top1_species'),
        'cv_top1_confidence': incident.get('cv_top1_confidence'),
        'cv_top2_species': incident.get('cv_top2_species'),
        'cv_top2_confidence': incident.get('cv_top2_confidence'),
        'llm_likely_species': incident.get('llm_likely_species'),
        'llm_species_confidence': incident.get('llm_species_confidence'),
        'reported_mobility': incident.get('reported_mobility'),
        'place_name': incident.get('place_name'),
    }
    severity_placeholder = {
        'incident_id': incident.get('incident_id'),
        'note': 'The severity prompt will produce this packet from the incident context.',
    }
    return {
        'species_group_prompt': render_prompt('species_group_classifier_few_shot', species_json=as_json(species_context)),
        'validation_prompt': render_prompt('report_validation_prompt', incident_packet_json=as_json(incident)),
        'duplicate_prompt': render_prompt('duplicate_matching_prompt', current_incident_json=as_json(incident), prior_candidates_json=as_json(prior_candidate_packet(incident['incident_id']))),
        'severity_prompt': render_prompt('severity_triage_prompt', incident_packet_json=as_json(incident)),
        'action_prompt': render_prompt('recommended_action_prompt', severity_packet_json=as_json(severity_placeholder)),
        'dispatch_prompt': render_prompt('dispatch_center_prompt', incident_packet_json=as_json(incident), rescue_center_directory_json=as_json(directory_packet())),
        'alert_card_prompt': render_prompt(
            'alert_card_generator',
            severity_level='{severity_level_from_severity_prompt}',
            recommended_action='{recommended_action_from_action_prompt}',
            dispatch_center_name='{dispatch_center_from_dispatch_prompt}',
            dispatch_hotline='{hotline_from_dispatch_prompt}',
            volunteer_allowed='{volunteer_allowed_from_action_prompt}',
            coordinator_review_required='{coordinator_review_required_from_action_prompt}',
            agent2_record_json=as_json(incident),
        ),
        'orchestrator_prompt': render_prompt('agent2_orchestration_prompt', incident_packet_json=as_json(incident), prior_candidates_json=as_json(prior_candidate_packet(incident['incident_id'])), rescue_center_directory_json=as_json(directory_packet())),
    }

selected_demo = demo_incidents.loc[1]
prompt_packet = build_prompt_first_packet(selected_demo)
list(prompt_packet.keys())

['species_group_prompt',
 'validation_prompt',
 'duplicate_prompt',
 'severity_prompt',
 'action_prompt',
 'dispatch_prompt',
 'alert_card_prompt',
 'orchestrator_prompt']

## 6. Species Classification Prompt Module

Species grouping can be handled by Gemini instead of Python taxonomy mapping.

In [6]:
print(prompt_packet['species_group_prompt'])

You are the LifeJacket Species Group Classifier.

Allowed labels only:
- pinniped
- cetacean
- sea_turtle
- unknown

Few-shot examples:
Example A:
Input: cv_top1_species='California sea lion', llm_likely_species='sea lion', reporter='seal-like animal with flippers'
Output: {"species_group":"pinniped","species_group_confidence":0.95,"evidence_used":["sea lion is a pinniped"],"uncertainty_notes":"low uncertainty","needs_human_species_review":false}

Example B:
Input: cv_top1_species='dolphin', llm_likely_species='dolphin', reporter='animal stranded in surf, long body, dorsal fin'
Output: {"species_group":"cetacean","species_group_confidence":0.92,"evidence_used":["dolphin is a cetacean"],"uncertainty_notes":"low uncertainty","needs_human_species_review":false}

Example C:
Input: cv_top1_species='unknown', llm_likely_species='unknown', reporter='large dark shape on beach'
Output: {"species_group":"unknown","species_group_confidence":0.2,"evidence_used":["species evidence insufficient"],"u

## 7. Report Validation Prompt Module

This module checks whether the structured Agent 1 report has enough information for triage, and identifies contradictions or missing data without inventing details.


In [7]:
print(prompt_packet['validation_prompt'][:3500])


You are LifeJacket Report Validation Module.

Decide whether this report is complete enough for triage. Use the structured data only. Do not invent facts.

Return JSON only with keys:
- case_quality_status: complete | review_needed | incomplete
- data_conflict_flag: boolean
- needs_more_info: boolean
- missing_info: list of strings
- validation_rationale: string

Guidance:
- incomplete means key location, timestamp, species group, or animal condition evidence is missing.
- review_needed means enough data exists, but there are conflicts or ambiguity.
- complete means no major missing or conflicting information.

Incident packet:
{
  "incident_id": "DEMO_002_DUPLICATE_CRITICAL",
  "report_timestamp": "2026-05-20T15:12:00Z",
  "latitude": 34.421,
  "longitude": -119.698,
  "place_name": "Santa Barbara Beach, CA",
  "cv_top1_species": "California sea lion",
  "cv_top1_confidence": 0.78,
  "reported_mobility": "limited",
  "reported_visible_wound": true,
  "reported_bleeding": true,
  "repo

## 8. Duplicate Matching Prompt Module

This module compares the current report against previous cases using location, time, and species evidence to identify possible repeat reports.


In [8]:
print(prompt_packet['duplicate_prompt'][:3500])


You are LifeJacket Duplicate Incident Matching Module.

Compare the current report against prior incident candidates. Decide whether it is a new incident, possible duplicate, probable duplicate, or not evaluable.

Use these signals:
- GPS proximity
- time proximity
- species group similarity
- same animal/photo/reporter clues if present

Return JSON only with keys:
- case_match_status: new_incident | possible_duplicate | probable_duplicate | not_evaluable
- matched_incident_id: string or null
- match_confidence: number from 0 to 1
- match_rationale: string
- uncertainty_notes: string

Current report:
{
  "incident_id": "DEMO_002_DUPLICATE_CRITICAL",
  "report_timestamp": "2026-05-20T15:12:00Z",
  "latitude": 34.421,
  "longitude": -119.698,
  "place_name": "Santa Barbara Beach, CA",
  "cv_top1_species": "California sea lion",
  "cv_top1_confidence": 0.78,
  "reported_mobility": "limited",
  "reported_visible_wound": true,
  "reported_bleeding": true,
  "reported_entanglement": false,
 

## 9. Severity Triage Prompt Module

This is one of the central prompt-engineering examples in Agent 2. It moves severity policy into a structured Gemini prompt and requires JSON output for auditability.


```text
You are LifeJacket Severity Triage Prompt, a constrained operational AI agent.

Assign severity_level using this policy:
0 = Info
1 = Monitor
2 = Respond
3 = Critical

Critical rules:
- entanglement -> Level 3 Critical
- wound + bleeding -> Level 3 Critical
- respiratory distress + unresponsive -> Level 3 Critical
- cetacean + mobility concern -> Level 3 Critical

Scoring considerations if no critical rule is triggered:
- injury
- wound
- bleeding
- mobility concern
- respiratory distress
- abnormal posture
- swelling
- near people
- near dogs
- data quality uncertainty

Do not use weather for this first prototype.

Return JSON only with keys:
- severity_level: 0 | 1 | 2 | 3
- severity_score: number
- severity_confidence: number from 0 to 1
- triage_reason_codes: list of strings
- triggered_rules: list of strings
- triage_rationale: string

Incident packet:
{incident_packet_json}
```

### Severity Prompt Preview

The severity policy is written directly into the prompt.

In [9]:
print(prompt_packet['severity_prompt'])

You are LifeJacket Severity Triage Module, operating within Agent 2.

Assign severity_level using this policy:
0 = Info
1 = Monitor
2 = Respond
3 = Critical

Critical rules:
- entanglement -> Level 3 Critical
- wound + bleeding -> Level 3 Critical
- respiratory distress + unresponsive -> Level 3 Critical
- cetacean + mobility concern -> Level 3 Critical

Scoring considerations if no critical rule is triggered:
- injury
- wound
- bleeding
- mobility concern
- respiratory distress
- abnormal posture
- swelling
- near people
- near dogs
- data quality uncertainty

Do not use weather for this first prototype.

Return JSON only with keys:
- severity_level: 0 | 1 | 2 | 3
- severity_score: number
- severity_confidence: number from 0 to 1
- triage_reason_codes: list of strings
- triggered_rules: list of strings
- triage_rationale: string

Incident packet:
{
  "incident_id": "DEMO_002_DUPLICATE_CRITICAL",
  "report_timestamp": "2026-05-20T15:12:00Z",
  "latitude": 34.421,
  "longitude": -119.69

## 10. Recommended Action Prompt Module

This module translates the triage result into an operational action, alert priority, volunteer permission, and coordinator review requirement.


In [10]:
print(prompt_packet['action_prompt'])


You are LifeJacket Recommended Action Module.

Map severity to action:
- Level 0 -> Guidance only
- Level 1 -> Volunteer monitor
- Level 2 -> Notify rescue center
- Level 3 -> Immediate escalation

Return JSON only with keys:
- recommended_action
- alert_type
- alert_priority
- volunteer_allowed
- coordinator_review_required
- rationale

Severity packet:
{
  "incident_id": "DEMO_002_DUPLICATE_CRITICAL",
  "note": "The severity prompt will produce this packet from the incident context."
}


## 11. Dispatch Selection Prompt Module

The model chooses from directory rows instead of hard-coded county routing.

In [11]:
print(prompt_packet['dispatch_prompt'][:3500])

You are LifeJacket Dispatch Center Selection Module.

Choose the best rescue center from the directory rows using:
- response area match
- species group
- live/dead response need
- response capabilities
- hotline/contact availability

Do not invent centers. Choose only from the provided directory rows.

Return JSON only with keys:
- dispatch_center_name
- dispatch_hotline
- dispatch_email
- dispatch_website
- dispatch_response_area
- dispatch_response_type
- dispatch_match_confidence: number from 0 to 1
- dispatch_match_reason
- dispatch_alternate_centers: list of strings

Incident packet:
{
  "incident_id": "DEMO_002_DUPLICATE_CRITICAL",
  "report_timestamp": "2026-05-20T15:12:00Z",
  "latitude": 34.421,
  "longitude": -119.698,
  "place_name": "Santa Barbara Beach, CA",
  "cv_top1_species": "California sea lion",
  "cv_top1_confidence": 0.78,
  "reported_mobility": "limited",
  "reported_visible_wound": true,
  "reported_bleeding": true,
  "reported_entanglement": false,
  "reported_

## 12. Alert Card And Guardrail Prompt Modules

The Alert Card Prompt Module creates application-facing copy from the structured decision record. The Guardrail Prompt Module and grounding check prevent unsupported claims or unsafe instructions from being displayed.

This pair is placed after triage, action, and dispatch because it must preserve those earlier decisions exactly.


In [12]:
required_schema = {
    'severity_level': '0 | 1 | 2 | 3',
    'recommended_action': 'Guidance only | Volunteer monitor | Notify rescue center | Immediate escalation',
    'dispatch_center_name': 'must be selected from provided directory rows',
    'alert_card_next_steps': 'must include safety language',
}

bad_output = 'This animal should recover if treated quickly, and bystanders can help move it.'

repair_prompt = render_prompt('json_schema_repair_prompt', required_schema_json=as_json(required_schema), invalid_output_text=bad_output)
grounding_prompt = render_prompt('claim_grounding_auditor', incident_packet_json=as_json(row_packet(selected_demo)), generated_text=bad_output)

print('--- JSON REPAIR PROMPT ---')
print(repair_prompt)
print('\n--- CLAIM GROUNDING PROMPT ---')
print(grounding_prompt)

--- JSON REPAIR PROMPT ---
You are a JSON repair step. The previous model output did not match the required schema.

Return valid JSON only. Preserve the meaning of the original output where possible. Do not add facts that are not present.

Required schema:
{
  "severity_level": "0 | 1 | 2 | 3",
  "recommended_action": "Guidance only | Volunteer monitor | Notify rescue center | Immediate escalation",
  "dispatch_center_name": "must be selected from provided directory rows",
  "alert_card_next_steps": "must include safety language"
}

Invalid output:
This animal should recover if treated quickly, and bystanders can help move it.

--- CLAIM GROUNDING PROMPT ---
You are a claim-grounding auditor for a safety-critical wildlife rescue app.

Compare generated text against the structured incident packet. Every factual claim in the generated text must be supported by the packet. Flag unsupported claims, softened safety language, or changed decisions.

Return JSON only with keys:
- grounded
- u

### Guardrail Failure Demo: Unsafe Card Is Held

This example is useful in a presentation because it demonstrates a failure case: Gemini-generated wording is not automatically trusted. Unsupported recovery claims or instructions telling bystanders to move an animal cause the card to be held for coordinator review.


In [13]:
unsafe_generated_alert = {
    'alert_card_title_generated': 'Wounded animal needs help',
    'alert_card_summary_generated': 'This animal should recover if treated quickly.',
    'alert_card_key_risks_generated': ['visible wound and bleeding'],
    'alert_card_missing_info_generated': [],
    'alert_card_next_steps_generated': ['Bystanders can help move it away from the tide.'],
}

mock_guardrail_failure = {
    'pass_guardrails': False,
    'violations': ['Instructs bystanders to move an animal instead of waiting for authorized responders.'],
    'missing_required_content': ['Keep distance.', 'Do not touch, feed, move, or pour water on the animal.'],
    'invented_or_unsupported_claims': ['The animal should recover if treated quickly.'],
    'corrected_copy_if_needed': 'Keep distance and keep people and dogs away. Do not touch, feed, move, or pour water on the animal. Authorized responders have been notified.',
}

mock_grounding_failure = {
    'grounded': False,
    'unsupported_claims': ['The animal should recover if treated quickly.'],
    'changed_decisions': [],
    'unsafe_language': ['Bystanders can help move it away from the tide.'],
    'recommended_action': 'Hold alert-card copy for coordinator review and use approved safety guidance.',
}

failed_card_display_status = 'held_for_coordinator_review'
{
    'generated_alert': unsafe_generated_alert,
    'guardrail_output': mock_guardrail_failure,
    'grounding_output': mock_grounding_failure,
    'display_status': failed_card_display_status,
}


{'generated_alert': {'alert_card_title_generated': 'Wounded animal needs help',
  'alert_card_summary_generated': 'This animal should recover if treated quickly.',
  'alert_card_key_risks_generated': ['visible wound and bleeding'],
  'alert_card_missing_info_generated': [],
  'alert_card_next_steps_generated': ['Bystanders can help move it away from the tide.']},
 'guardrail_output': {'pass_guardrails': False,
  'violations': ['Instructs bystanders to move an animal instead of waiting for authorized responders.'],
  'missing_required_content': ['Keep distance.',
   'Do not touch, feed, move, or pour water on the animal.'],
  'invented_or_unsupported_claims': ['The animal should recover if treated quickly.'],
  'corrected_copy_if_needed': 'Keep distance and keep people and dogs away. Do not touch, feed, move, or pour water on the animal. Authorized responders have been notified.'},
 'grounding_output': {'grounded': False,
  'unsupported_claims': ['The animal should recover if treated qu

### Example Approved Case Records

These mock approved case records make the end-of-day documentation example runnable without a live Vertex AI call. In the app, these records would come from the validated live chain.

In [14]:
mock_outputs = pd.DataFrame([
    {
        'incident_id': 'DEMO_001_PINNIPED_WOUND',
        'species_group': 'pinniped',
        'case_quality_status': 'complete',
        'case_match_status': 'new_incident',
        'severity_level': 1,
        'recommended_action': 'Volunteer monitor',
        'dispatch_center_name': 'Channel Islands Marine & Wildlife Institute (CIMWI)',
        'alert_card_title': 'Monitor: California sea lion near Santa Barbara Beach',
    },
    {
        'incident_id': 'DEMO_002_DUPLICATE_CRITICAL',
        'species_group': 'pinniped',
        'case_quality_status': 'complete',
        'case_match_status': 'probable_duplicate',
        'severity_level': 3,
        'recommended_action': 'Immediate escalation',
        'dispatch_center_name': 'Channel Islands Marine & Wildlife Institute (CIMWI)',
        'alert_card_title': 'Critical: wounded sea lion near Santa Barbara Beach',
    },
    {
        'incident_id': 'DEMO_003_CETACEAN_MOBILITY',
        'species_group': 'cetacean',
        'case_quality_status': 'review_needed',
        'case_match_status': 'new_incident',
        'severity_level': 3,
        'recommended_action': 'Immediate escalation',
        'dispatch_center_name': 'Channel Islands Marine & Wildlife Institute (CIMWI)',
        'alert_card_title': 'Critical: dolphin mobility concern near Ventura Beach',
    },
    {
        'incident_id': 'DEMO_004_INCOMPLETE_UNKNOWN',
        'species_group': 'unknown',
        'case_quality_status': 'incomplete',
        'case_match_status': 'not_evaluable',
        'severity_level': 0,
        'recommended_action': 'Guidance only',
        'dispatch_center_name': '',
        'alert_card_title': 'Info: incomplete wildlife report needs follow-up',
    },
    {
        'incident_id': 'DEMO_005_TURTLE_ENTANGLEMENT',
        'species_group': 'sea_turtle',
        'case_quality_status': 'complete',
        'case_match_status': 'new_incident',
        'severity_level': 3,
        'recommended_action': 'Immediate escalation',
        'dispatch_center_name': 'Pacific Marine Mammal Center (PMMC)',
        'alert_card_title': 'Critical: entangled sea turtle near Laguna Beach',
    },
])

mock_outputs

,incident_id,species_group,case_quality_status,case_match_status,severity_level,recommended_action,dispatch_center_name,alert_card_title
0,DEMO_001_PINNIPED_WOUND,pinniped,complete,new_incident,1,Volunteer monitor,Channel Islands Marine & Wildlife Institute (CIMWI),Monitor: California sea lion near Santa Barbara Beach
1,DEMO_002_DUPLICATE_CRITICAL,pinniped,complete,probable_duplicate,3,Immediate escalation,Channel Islands Marine & Wildlife Institute (CIMWI),Critical: wounded sea lion near Santa Barbara Beach
2,DEMO_003_CETACEAN_MOBILITY,cetacean,review_needed,new_incident,3,Immediate escalation,Channel Islands Marine & Wildlife Institute (CIMWI),Critical: dolphin mobility concern near Ventura Beach
3,DEMO_004_INCOMPLETE_UNKNOWN,unknown,incomplete,not_evaluable,0,Guidance only,,Info: incomplete wildlife report needs follow-up
4,DEMO_005_TURTLE_ENTANGLEMENT,sea_turtle,complete,new_incident,3,Immediate escalation,Pacific Marine Mammal Center (PMMC),Critical: entangled sea turtle near Laguna Beach


## 13. End-of-Day Documentation Prompt Module

At shift close, the app can automatically send every **approved Agent 2 case record** to a documentation module. This creates a daily operational log without allowing Gemini to change triage or dispatch decisions.

The documentation layer preserves incomplete and unresolved cases so a coordinator can carry them into the next shift. In a deployed app, this step is triggered by a scheduled end-of-day workflow and stores both the source case rows and generated report JSON for audit.


In [15]:
daily_case_records = demo_incidents[['incident_id', 'report_timestamp', 'place_name']].merge(
    mock_outputs, on='incident_id', how='left'
).sort_values('report_timestamp')

DAILY_CASE_LOG_CSV = Path('agent2_daily_case_log_demo.csv')
daily_case_records.to_csv(DAILY_CASE_LOG_CSV, index=False)
print(f'Structured daily case log written to: {DAILY_CASE_LOG_CSV}')
daily_case_records


Structured daily case log written to: agent2_daily_case_log_demo.csv


,incident_id,report_timestamp,place_name,species_group,case_quality_status,case_match_status,severity_level,recommended_action,dispatch_center_name,alert_card_title
0,DEMO_001_PINNIPED_WOUND,2026-05-20T15:00:00Z,"Santa Barbara Beach, CA",pinniped,complete,new_incident,1,Volunteer monitor,Channel Islands Marine & Wildlife Institute (CIMWI),Monitor: California sea lion near Santa Barbara Beach
1,DEMO_002_DUPLICATE_CRITICAL,2026-05-20T15:12:00Z,"Santa Barbara Beach, CA",pinniped,complete,probable_duplicate,3,Immediate escalation,Channel Islands Marine & Wildlife Institute (CIMWI),Critical: wounded sea lion near Santa Barbara Beach
2,DEMO_003_CETACEAN_MOBILITY,2026-05-20T16:30:00Z,"Ventura Beach, CA",cetacean,review_needed,new_incident,3,Immediate escalation,Channel Islands Marine & Wildlife Institute (CIMWI),Critical: dolphin mobility concern near Ventura Beach
3,DEMO_004_INCOMPLETE_UNKNOWN,2026-05-20T17:05:00Z,unknown beach,unknown,incomplete,not_evaluable,0,Guidance only,,Info: incomplete wildlife report needs follow-up
4,DEMO_005_TURTLE_ENTANGLEMENT,2026-05-20T18:00:00Z,"Laguna Beach, Orange County, CA",sea_turtle,complete,new_incident,3,Immediate escalation,Pacific Marine Mammal Center (PMMC),Critical: entangled sea turtle near Laguna Beach


### Documentation Prompt Preview

This prompt is deliberately strict: it may summarize the day, but it may not invent outcomes or overwrite approved decisions.


In [16]:
daily_documentation_prompt = render_prompt(
    'end_of_day_documentation_prompt',
    daily_case_records_json=as_json(daily_case_records.to_dict(orient='records')),
)
print(daily_documentation_prompt)


You are the LifeJacket End-of-Day Case Documentation Module.

Create the daily operational record from approved case outputs. Every incident must be documented, including informational and incomplete cases.

Non-negotiable constraints:
- Use only the approved case records provided below.
- Do not change severity, recommended action, duplicate status, dispatch center, or hotline.
- Do not invent response outcomes, animal outcomes, or responder actions.
- Preserve unresolved and missing-information cases for next-shift review.
- If information is missing, state that it is not recorded.

Return JSON only with keys:
- report_date
- total_cases_documented
- severity_counts
- critical_incidents
- dispatches_recorded
- unresolved_or_followup_cases
- duplicate_incident_notes
- chronological_case_log
- next_shift_attention_items
- audit_statement

Each chronological_case_log item must include:
- incident_id
- location
- species_group
- severity_level
- recommended_action
- dispatch_center_name


### Mock Daily Documentation Output

This mocked JSON demonstrates the file the future app could archive each evening. A live deployment would replace it with Gemini JSON after schema and grounding checks.


In [17]:
mock_end_of_day_documentation = {
    'report_date': '2026-05-20',
    'total_cases_documented': 5,
    'severity_counts': {'0_info': 1, '1_monitor': 1, '2_respond': 0, '3_critical': 3},
    'critical_incidents': [
        'DEMO_002_DUPLICATE_CRITICAL',
        'DEMO_003_CETACEAN_MOBILITY',
        'DEMO_005_TURTLE_ENTANGLEMENT',
    ],
    'dispatches_recorded': [
        {'incident_id': 'DEMO_002_DUPLICATE_CRITICAL', 'center': 'Channel Islands Marine & Wildlife Institute (CIMWI)'},
        {'incident_id': 'DEMO_003_CETACEAN_MOBILITY', 'center': 'Channel Islands Marine & Wildlife Institute (CIMWI)'},
        {'incident_id': 'DEMO_005_TURTLE_ENTANGLEMENT', 'center': 'Pacific Marine Mammal Center (PMMC)'},
    ],
    'unresolved_or_followup_cases': ['DEMO_004_INCOMPLETE_UNKNOWN'],
    'duplicate_incident_notes': ['DEMO_002_DUPLICATE_CRITICAL is recorded as a probable duplicate.'],
    'next_shift_attention_items': ['Obtain missing location and condition details for DEMO_004_INCOMPLETE_UNKNOWN.'],
    'audit_statement': 'Summary derived only from approved demo case records; no outcomes are inferred.',
}
DAILY_DOCUMENTATION_JSON = Path('agent2_end_of_day_report_demo.json')
DAILY_DOCUMENTATION_JSON.write_text(as_json(mock_end_of_day_documentation) + '\n')
print(f'Daily documentation report written to: {DAILY_DOCUMENTATION_JSON}')
mock_end_of_day_documentation


Daily documentation report written to: agent2_end_of_day_report_demo.json


{'report_date': '2026-05-20',
 'total_cases_documented': 5,
 'severity_counts': {'0_info': 1,
  '1_monitor': 1,
  '2_respond': 0,
  '3_critical': 3},
 'critical_incidents': ['DEMO_002_DUPLICATE_CRITICAL',
  'DEMO_003_CETACEAN_MOBILITY',
  'DEMO_005_TURTLE_ENTANGLEMENT'],
 'dispatches_recorded': [{'incident_id': 'DEMO_002_DUPLICATE_CRITICAL',
   'center': 'Channel Islands Marine & Wildlife Institute (CIMWI)'},
  {'incident_id': 'DEMO_003_CETACEAN_MOBILITY',
   'center': 'Channel Islands Marine & Wildlife Institute (CIMWI)'},
  {'incident_id': 'DEMO_005_TURTLE_ENTANGLEMENT',
   'center': 'Pacific Marine Mammal Center (PMMC)'}],
 'unresolved_or_followup_cases': ['DEMO_004_INCOMPLETE_UNKNOWN'],
 'duplicate_incident_notes': ['DEMO_002_DUPLICATE_CRITICAL is recorded as a probable duplicate.'],
 'next_shift_attention_items': ['Obtain missing location and condition details for DEMO_004_INCOMPLETE_UNKNOWN.'],
 'audit_statement': 'Summary derived only from approved demo case records; no outcomes

## 14. Agent 2 Orchestration Prompt Module

This prompt describes the complete multi-prompt workflow.

In [18]:
print(prompt_packet['orchestrator_prompt'][:3500])

You are the LifeJacket Agent 2 Prompt Orchestrator.

Run the conceptual sequence below and return the final assembled decision packet.

Prompt stages:
1. species_group_classifier_few_shot
2. report_validation_prompt
3. duplicate_matching_prompt
4. severity_triage_prompt
5. recommended_action_prompt
6. dispatch_center_prompt
7. alert_card_generator
8. alert_card_guardrail_critic
9. claim_grounding_auditor
10. audit_log_narrator

Rules:
- Each prompt stage returns JSON.
- Later prompt stages must use earlier outputs.
- Any invalid JSON should be sent to json_schema_repair_prompt.
- Any generated UI copy must pass alert_card_guardrail_critic and claim_grounding_auditor.
- Unsafe or unsupported copy must be held for coordinator review, not displayed.

Return JSON only with keys:
- species_output
- validation_output
- duplicate_output
- severity_output
- action_output
- dispatch_output
- alert_card_output
- guardrail_output
- grounding_output
- audit_output
- display_status
- final_decision

## 15. Live Vertex AI Gemini Chaining

The earlier cells build prompt packets. This section runs the **actual Agent 2 chain** through Gemini on Vertex AI: each prompt module returns structured JSON, and its output becomes context for the next module.

The notebook uses the Google Gen AI SDK with Vertex AI and JSON-schema constrained outputs. In your GCP JupyterLab terminal, set the project configuration before starting or restarting the notebook kernel:

```bash
export GOOGLE_CLOUD_PROJECT="your-gcp-project-id"
export GOOGLE_CLOUD_LOCATION="global"
export GOOGLE_GENAI_USE_VERTEXAI="True"
export VERTEX_GEMINI_MODEL="publishers/google/models/gemini-3.5-flash"
```

Authentication comes from your Google Cloud environment, such as the service account or Application Default Credentials configured for the JupyterLab runtime. No Gemini API key is stored in the notebook.


### 15.1 Vertex AI Configuration


In [19]:
import os

GOOGLE_CLOUD_PROJECT = os.getenv('GOOGLE_CLOUD_PROJECT', '')
GOOGLE_CLOUD_LOCATION = os.getenv('GOOGLE_CLOUD_LOCATION', 'global')
os.environ['GOOGLE_GENAI_USE_VERTEXAI'] = 'True'
VERTEX_GEMINI_MODEL = os.getenv('VERTEX_GEMINI_MODEL', 'publishers/google/models/gemini-3.5-flash')
GEMINI_TEMPERATURE = 0.1

print('Vertex AI project configured:', bool(GOOGLE_CLOUD_PROJECT))
print('Vertex AI location:', GOOGLE_CLOUD_LOCATION)
print('Vertex AI Gemini model:', VERTEX_GEMINI_MODEL)


Vertex AI project configured: True
Vertex AI location: global
Vertex AI Gemini model: publishers/google/models/gemini-3.5-flash


### 15.2 Structured Output Schemas

These schemas make the chain auditable: outputs are parsed as JSON and restricted to fields the application expects.


In [20]:
def json_object_schema(properties):
    return {'type': 'object', 'properties': properties, 'required': list(properties)}

STRING = {'type': 'string'}
STRING_LIST = {'type': 'array', 'items': {'type': 'string'}}
NUMBER_01 = {'type': 'number', 'minimum': 0, 'maximum': 1}

GEMINI_SCHEMAS = {
    'species': json_object_schema({
        'species_group': {'type': 'string', 'enum': ['pinniped', 'cetacean', 'sea_turtle', 'unknown']},
        'species_group_confidence': NUMBER_01,
        'evidence_used': STRING_LIST,
        'uncertainty_notes': STRING,
        'needs_human_species_review': {'type': 'boolean'},
    }),
    'validation': json_object_schema({
        'case_quality_status': {'type': 'string', 'enum': ['complete', 'review_needed', 'incomplete']},
        'data_conflict_flag': {'type': 'boolean'},
        'needs_more_info': {'type': 'boolean'},
        'missing_info': STRING_LIST,
        'validation_rationale': STRING,
    }),
    'duplicate': json_object_schema({
        'case_match_status': {'type': 'string', 'enum': ['new_incident', 'possible_duplicate', 'probable_duplicate', 'not_evaluable']},
        'matched_incident_id': {'type': ['string', 'null']},
        'match_confidence': NUMBER_01,
        'match_rationale': STRING,
        'uncertainty_notes': STRING,
    }),
    'severity': json_object_schema({
        'severity_level': {'type': 'integer', 'enum': [0, 1, 2, 3]},
        'severity_score': {'type': 'number'},
        'severity_confidence': NUMBER_01,
        'triage_reason_codes': STRING_LIST,
        'triggered_rules': STRING_LIST,
        'triage_rationale': STRING,
    }),
    'action': json_object_schema({
        'recommended_action': {'type': 'string', 'enum': ['Guidance only', 'Volunteer monitor', 'Notify rescue center', 'Immediate escalation']},
        'alert_type': STRING,
        'alert_priority': STRING,
        'volunteer_allowed': {'type': 'boolean'},
        'coordinator_review_required': {'type': 'boolean'},
        'rationale': STRING,
    }),
    'dispatch': json_object_schema({
        'dispatch_center_name': STRING,
        'dispatch_hotline': STRING,
        'dispatch_email': STRING,
        'dispatch_website': STRING,
        'dispatch_response_area': STRING,
        'dispatch_response_type': STRING,
        'dispatch_match_confidence': NUMBER_01,
        'dispatch_match_reason': STRING,
        'dispatch_alternate_centers': STRING_LIST,
    }),
    'alert_card': json_object_schema({
        'alert_card_title_generated': STRING,
        'alert_card_summary_generated': STRING,
        'alert_card_key_risks_generated': STRING_LIST,
        'alert_card_missing_info_generated': STRING_LIST,
        'alert_card_next_steps_generated': STRING_LIST,
    }),
    'card_guardrail': json_object_schema({
        'pass_guardrails': {'type': 'boolean'},
        'violations': STRING_LIST,
        'missing_required_content': STRING_LIST,
        'invented_or_unsupported_claims': STRING_LIST,
        'corrected_copy_if_needed': STRING,
    }),
    'grounding': json_object_schema({
        'grounded': {'type': 'boolean'},
        'unsupported_claims': STRING_LIST,
        'changed_decisions': STRING_LIST,
        'unsafe_language': STRING_LIST,
        'recommended_action': STRING,
    }),
    'audit': json_object_schema({
        'audit_summary': STRING,
        'inputs_used': STRING_LIST,
        'rules_triggered': STRING_LIST,
        'outputs_generated': STRING_LIST,
        'residual_risks': STRING_LIST,
    }),
    'prompt_review': json_object_schema({
        'prompt_strengths': STRING_LIST,
        'prompt_weaknesses': STRING_LIST,
        'safety_requirements_preserved': STRING_LIST,
        'safety_requirements_missing': STRING_LIST,
        'ambiguous_instructions': STRING_LIST,
        'recommended_revision_goals': STRING_LIST,
        'review_confidence': NUMBER_01,
    }),
    'prompt_revision': json_object_schema({
        'revised_prompt': STRING,
        'changes_made': STRING_LIST,
        'preserved_safety_requirements': STRING_LIST,
        'remaining_risks': STRING_LIST,
        'requires_human_approval': {'type': 'boolean'},
    }),
    'prompt_comparison': json_object_schema({
        'comparison_summary': STRING,
        'baseline_risks': STRING_LIST,
        'revised_prompt_improvements': STRING_LIST,
        'revised_prompt_regressions': STRING_LIST,
        'safety_requirements_preserved': STRING_LIST,
        'recommendation': {'type': 'string', 'enum': ['promote_after_human_review', 'revise_again', 'reject']},
        'human_review_checklist': STRING_LIST,
    }),
    'daily_documentation': json_object_schema({
        'report_date': STRING,
        'total_cases_documented': {'type': 'integer'},
        'severity_counts': json_object_schema({
            '0_info': {'type': 'integer'},
            '1_monitor': {'type': 'integer'},
            '2_respond': {'type': 'integer'},
            '3_critical': {'type': 'integer'},
        }),
        'critical_incidents': STRING_LIST,
        'dispatches_recorded': {'type': 'array', 'items': json_object_schema({
            'incident_id': STRING,
            'dispatch_center_name': STRING,
            'dispatch_hotline': STRING,
        })},
        'unresolved_or_followup_cases': STRING_LIST,
        'duplicate_incident_notes': STRING_LIST,
        'chronological_case_log': {'type': 'array', 'items': json_object_schema({
            'incident_id': STRING,
            'location': STRING,
            'species_group': STRING,
            'severity_level': {'type': 'integer', 'enum': [0, 1, 2, 3]},
            'recommended_action': STRING,
            'dispatch_center_name': STRING,
            'case_status_summary': STRING,
            'documentation_limitations': STRING,
        })},
        'next_shift_attention_items': STRING_LIST,
        'audit_statement': STRING,
    }),
}


### 15.3 Vertex AI Gemini Transport Function

This is the only model transport function. It sends one prompt module to Vertex AI Gemini and receives schema-constrained JSON.


In [21]:
def call_vertex_gemini_json(prompt, response_schema, temperature=GEMINI_TEMPERATURE):
    """Send one Agent 2 prompt module to Gemini on Vertex AI and return JSON."""
    from google import genai
    from google.genai.types import HttpOptions

    assert GOOGLE_CLOUD_PROJECT, 'Set GOOGLE_CLOUD_PROJECT before running live Vertex AI Gemini cells.'
    client = genai.Client(
        vertexai=True,
        project=GOOGLE_CLOUD_PROJECT,
        location=GOOGLE_CLOUD_LOCATION,
        http_options=HttpOptions(api_version='v1'),
    )
    response = client.models.generate_content(
        model=VERTEX_GEMINI_MODEL,
        contents=prompt,
        config={
            'temperature': temperature,
            'response_mime_type': 'application/json',
            'response_json_schema': response_schema,
        },
    )
    return json.loads(response.text)


### 15.4 One Incident Through The Full Chain

The context grows step by step. This makes the chaining visible: species output feeds validation and triage; triage feeds the action prompt; the action and dispatch outputs constrain the alert card; two validation prompts determine whether card copy may be displayed.


In [22]:
def run_live_agent2_chain(row):
    incident = row_packet(row)
    outputs = {}

    species_input = {key: incident.get(key) for key in [
        'cv_top1_species', 'cv_top1_confidence', 'cv_top2_species', 'cv_top2_confidence',
        'llm_likely_species', 'llm_species_confidence', 'reported_mobility', 'place_name'
    ]}
    outputs['species_output'] = call_vertex_gemini_json(
        render_prompt('species_group_classifier_few_shot', species_json=as_json(species_input)),
        GEMINI_SCHEMAS['species'],
    )
    decision_record = {**incident, **outputs['species_output']}

    outputs['validation_output'] = call_vertex_gemini_json(
        render_prompt('report_validation_prompt', incident_packet_json=as_json(decision_record)),
        GEMINI_SCHEMAS['validation'],
    )
    decision_record.update(outputs['validation_output'])

    outputs['duplicate_output'] = call_vertex_gemini_json(
        render_prompt(
            'duplicate_matching_prompt',
            current_incident_json=as_json(decision_record),
            prior_candidates_json=as_json(prior_candidate_packet(incident['incident_id'])),
        ),
        GEMINI_SCHEMAS['duplicate'],
    )
    decision_record.update(outputs['duplicate_output'])

    outputs['severity_output'] = call_vertex_gemini_json(
        render_prompt('severity_triage_prompt', incident_packet_json=as_json(decision_record)),
        GEMINI_SCHEMAS['severity'],
    )
    decision_record.update(outputs['severity_output'])

    outputs['action_output'] = call_vertex_gemini_json(
        render_prompt('recommended_action_prompt', severity_packet_json=as_json(decision_record)),
        GEMINI_SCHEMAS['action'],
    )
    decision_record.update(outputs['action_output'])

    outputs['dispatch_output'] = call_vertex_gemini_json(
        render_prompt(
            'dispatch_center_prompt',
            incident_packet_json=as_json(decision_record),
            rescue_center_directory_json=as_json(directory_packet(rows=len(rescue_centers))),
        ),
        GEMINI_SCHEMAS['dispatch'],
    )
    decision_record.update(outputs['dispatch_output'])

    outputs['alert_card_output'] = call_vertex_gemini_json(
        render_prompt(
            'alert_card_generator',
            severity_level=decision_record['severity_level'],
            recommended_action=decision_record['recommended_action'],
            dispatch_center_name=decision_record['dispatch_center_name'],
            dispatch_hotline=decision_record['dispatch_hotline'],
            volunteer_allowed=decision_record['volunteer_allowed'],
            coordinator_review_required=decision_record['coordinator_review_required'],
            agent2_record_json=as_json(decision_record),
        ),
        GEMINI_SCHEMAS['alert_card'],
    )

    outputs['guardrail_output'] = call_vertex_gemini_json(
        render_prompt(
            'alert_card_guardrail_critic',
            agent2_record_json=as_json(decision_record),
            generated_alert_json=as_json(outputs['alert_card_output']),
        ),
        GEMINI_SCHEMAS['card_guardrail'],
    )
    outputs['grounding_output'] = call_vertex_gemini_json(
        render_prompt(
            'claim_grounding_auditor',
            incident_packet_json=as_json(decision_record),
            generated_text=as_json(outputs['alert_card_output']),
        ),
        GEMINI_SCHEMAS['grounding'],
    )
    outputs['audit_output'] = call_vertex_gemini_json(
        render_prompt('audit_log_narrator', agent2_record_json=as_json({**decision_record, **outputs})),
        GEMINI_SCHEMAS['audit'],
    )

    outputs['display_approved'] = all([
        outputs['guardrail_output']['pass_guardrails'],
        outputs['grounding_output']['grounded'],
    ])
    outputs['display_status'] = {
        True: 'approved_for_display',
        False: 'held_for_coordinator_review',
    }[outputs['display_approved']]
    outputs['decision_record'] = decision_record
    return outputs


### 15.5 Run A Live Demo Case

After configuring Vertex AI authentication and `GOOGLE_CLOUD_PROJECT`, uncomment and run these lines. The output includes every chained JSON response and the final card-display approval status.


In [23]:
# live_demo_result = run_live_agent2_chain(selected_demo)
# Path('agent2_live_chain_demo.json').write_text(as_json(live_demo_result) + '\n')
# live_demo_result['display_status'], live_demo_result['decision_record']


### 15.6 Process Agent 1 CSV And Create Daily Documentation

This is the app-ready execution flow: process each Agent 1 row through the live chain, save the case decision records, then use the end-of-day documentation prompt across the full approved case log.


In [24]:
def run_live_end_of_day_documentation(case_records):
    documentation_prompt = render_prompt(
        'end_of_day_documentation_prompt',
        daily_case_records_json=as_json(case_records),
    )
    return call_vertex_gemini_json(documentation_prompt, GEMINI_SCHEMAS['daily_documentation'])

# Uncomment these lines after configuring Vertex AI access.
# agent1_reports = pd.read_csv(AGENT1_INPUT_CSV).fillna('')
# live_case_results = [run_live_agent2_chain(row) for _, row in agent1_reports.iterrows()]
# live_case_records = [result['decision_record'] for result in live_case_results]
# pd.DataFrame(live_case_records).to_csv('agent2_live_decision_output.csv', index=False)
# live_daily_report = run_live_end_of_day_documentation(live_case_records)
# Path('agent2_live_end_of_day_report.json').write_text(as_json(live_daily_report) + '\n')
# live_daily_report


## 16. Prompt Improvement Module

Agent 2 can improve its prompt modules through a controlled evaluation loop:

1. Run a candidate prompt against fixed evaluation scenarios.
2. Ask Gemini to identify ambiguities and missing safety constraints.
3. Ask Gemini to propose a revised prompt while preserving required rules and JSON outputs.
4. Compare the baseline and revision on the same scenarios.
5. Record the iteration and require human approval before promotion.

This keeps prompt engineering visible and auditable. The live system does **not** silently rewrite the operational prompt used for a case.


### 16.1 Fixed Evaluation Scenarios

These evaluation anchors are intentionally separate from real incident processing. They test whether a prompt preserves the required safety behavior.


In [25]:
iteration_evaluation_cases = [
    {
        'case_id': 'ITER_001_ENTANGLEMENT',
        'facts': {'species_group': 'pinniped', 'llm_entanglement_present': True},
        'expected_anchor': 'Must trigger Level 3 Critical for entanglement.',
    },
    {
        'case_id': 'ITER_002_WOUND_BLEEDING',
        'facts': {'llm_wound_present': True, 'llm_bleeding_present': True},
        'expected_anchor': 'Must trigger Level 3 Critical for wound plus bleeding.',
    },
    {
        'case_id': 'ITER_003_CETACEAN_MOBILITY',
        'facts': {'species_group': 'cetacean', 'llm_mobility_concern': 'yes'},
        'expected_anchor': 'Must trigger Level 3 Critical for cetacean mobility concern.',
    },
    {
        'case_id': 'ITER_004_INCOMPLETE',
        'facts': {'species_group': 'unknown', 'place_name': 'unknown beach'},
        'expected_anchor': 'Must express uncertainty and must not invent facts.',
    },
]

baseline_severity_prompt = prompt_index['severity_triage_prompt']['template']
quality_review_request = render_prompt(
    'prompt_quality_review_prompt',
    candidate_prompt=baseline_severity_prompt,
    evaluation_cases_json=as_json(iteration_evaluation_cases),
)
print(quality_review_request[:3500])


You are the LifeJacket Prompt Quality Review Module.

Evaluate the candidate prompt for clarity, completeness, auditability, and safety using the provided evaluation cases. You are reviewing the prompt, not deciding a real incident.

Mandatory safety requirements that must remain explicit:
- entanglement -> Level 3 Critical
- wound plus bleeding -> Level 3 Critical
- respiratory distress plus unresponsive -> Level 3 Critical
- cetacean with mobility concern -> Level 3 Critical
- weather is excluded from this first prototype
- JSON output schema must remain explicit
- no invented facts, outcomes, or responder actions

Return JSON only with keys:
- prompt_strengths
- prompt_weaknesses
- safety_requirements_preserved
- safety_requirements_missing
- ambiguous_instructions
- recommended_revision_goals
- review_confidence

Candidate prompt:
You are LifeJacket Severity Triage Module, operating within Agent 2.

Assign severity_level using this policy:
0 = Info
1 = Monitor
2 = Respond
3 = Criti

### 16.2 Live Review, Revision, And Comparison Chain

When Vertex AI access is configured, this function runs one documented improvement iteration. It returns a proposal and a comparison record, but does not automatically replace the baseline prompt.


In [26]:
def run_prompt_improvement_iteration(candidate_prompt, evaluation_cases):
    review_output = call_vertex_gemini_json(
        render_prompt(
            'prompt_quality_review_prompt',
            candidate_prompt=candidate_prompt,
            evaluation_cases_json=as_json(evaluation_cases),
        ),
        GEMINI_SCHEMAS['prompt_review'],
    )
    revision_output = call_vertex_gemini_json(
        render_prompt(
            'prompt_revision_prompt',
            candidate_prompt=candidate_prompt,
            quality_review_json=as_json(review_output),
        ),
        GEMINI_SCHEMAS['prompt_revision'],
    )
    comparison_output = call_vertex_gemini_json(
        render_prompt(
            'prompt_iteration_comparison_prompt',
            baseline_prompt=candidate_prompt,
            revised_prompt=revision_output['revised_prompt'],
            evaluation_cases_json=as_json(evaluation_cases),
        ),
        GEMINI_SCHEMAS['prompt_comparison'],
    )
    return {
        'baseline_prompt_id': 'severity_triage_prompt',
        'review_output': review_output,
        'revision_output': revision_output,
        'comparison_output': comparison_output,
        'promotion_status': 'requires_human_approval',
    }

# Uncomment after configuring Vertex AI access.
# live_prompt_iteration = run_prompt_improvement_iteration(baseline_severity_prompt, iteration_evaluation_cases)
# Path('agent2_live_prompt_iteration.json').write_text(as_json(live_prompt_iteration) + '\n')
# live_prompt_iteration


### 16.3 Mock Prompt Iteration Record

This offline example shows the type of improvement record that can be saved for an evaluator or future app administrator.


In [27]:
mock_prompt_iteration = {
    'baseline_prompt_id': 'severity_triage_prompt',
    'iteration_number': 1,
    'review_output': {
        'prompt_strengths': ['Critical escalation rules are explicit.', 'JSON fields support auditability.'],
        'prompt_weaknesses': ['Non-critical scoring considerations do not describe how uncertainty should be discussed.'],
        'safety_requirements_missing': [],
        'recommended_revision_goals': ['Require the rationale to state uncertain or missing evidence.'],
    },
    'revision_output': {
        'changes_made': ['Added an instruction to identify missing evidence in triage_rationale.'],
        'preserved_safety_requirements': ['All four critical rules', 'JSON-only output', 'Weather excluded'],
        'requires_human_approval': True,
    },
    'comparison_output': {
        'comparison_summary': 'Revision improves uncertainty reporting without changing critical escalation behavior.',
        'recommendation': 'promote_after_human_review',
        'human_review_checklist': ['Confirm critical rules remain unchanged.', 'Run all fixed evaluation scenarios before approval.'],
    },
    'promotion_status': 'requires_human_approval',
}

PROMPT_ITERATION_DEMO_JSON = Path('agent2_prompt_iteration_demo.json')
PROMPT_ITERATION_DEMO_JSON.write_text(as_json(mock_prompt_iteration) + '\n')
print(f'Prompt iteration record written to: {PROMPT_ITERATION_DEMO_JSON}')
mock_prompt_iteration


Prompt iteration record written to: agent2_prompt_iteration_demo.json


{'baseline_prompt_id': 'severity_triage_prompt',
 'iteration_number': 1,
 'review_output': {'prompt_strengths': ['Critical escalation rules are explicit.',
   'JSON fields support auditability.'],
  'prompt_weaknesses': ['Non-critical scoring considerations do not describe how uncertainty should be discussed.'],
  'safety_requirements_missing': [],
  'recommended_revision_goals': ['Require the rationale to state uncertain or missing evidence.']},
 'revision_output': {'changes_made': ['Added an instruction to identify missing evidence in triage_rationale.'],
  'preserved_safety_requirements': ['All four critical rules',
   'JSON-only output',
   'Weather excluded'],
  'requires_human_approval': True},
 'comparison_output': {'comparison_summary': 'Revision improves uncertainty reporting without changing critical escalation behavior.',
  'recommendation': 'promote_after_human_review',
  'human_review_checklist': ['Confirm critical rules remain unchanged.',
   'Run all fixed evaluation sce

## 17. App Architecture

```mermaid
flowchart TD
    A["Agent 1: Structured incident facts"] --> B["Agent 2: Prompt orchestrator"]
    B --> C["Decision prompt stages"]
    C --> D["Alert card prompt"]
    D --> E["Guardrail and grounding audit"]
    E --> F["Approved alert card and case record"]
    F --> G["End-of-day documentation module"]
    G --> H["Daily case log and next-shift handoff"]
    P["Fixed prompt evaluation scenarios"] --> Q["Review, revision, comparison"]
    Q --> R["Human approval before promotion"]
    R -. approved prompt version .-> B
```

In the final app, the pipeline can look like this:

1. Agent 1 extracts structured report facts.
2. Agent 2 prompt orchestrator builds specialist prompts.
3. Gemini prompt stages return structured JSON.
4. Python validates JSON schemas and allowed values.
5. Guardrail prompts check for unsupported claims or decision drift.
6. The app displays approved alert cards and stores every prompt/output for audit.
7. At shift close, the documentation module creates an archived daily case log and flags unresolved follow-up work.
8. Outside active case processing, prompt iterations are evaluated and approved before a new version can be used.

This is why Python still exists, but not as the main decision-policy layer.
